## 📦 1. Kurulum

In [2]:
import subprocess, sys

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "-q"],
    capture_output=True, text=True
)
print("pylibjpeg:", "OK" if r.returncode == 0 else r.stderr[-200:])

r2 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "gdcm", "-q"],
    capture_output=True, text=True
)
print("gdcm:", "OK" if r2.returncode == 0 else r2.stderr[-200:])

import importlib, pydicom
importlib.reload(pydicom)
pydicom.config.pixel_data_handlers = ["gdcm", "pylibjpeg", "numpy"]

# Test
from pathlib import Path
test = list(Path("/kaggle/input/datasets/crawford/qureai-headct/qct01").rglob("*.dcm"))[:1]
ds = pydicom.dcmread(str(test[0]))
print(ds.pixel_array.shape)
print("✅ DICOM okuma çalışıyor")

pylibjpeg: OK
gdcm: OK
(512, 512)
✅ DICOM okuma çalışıyor


In [43]:
%%capture
# ── Temel kütüphaneler ───────────────────────────────────────
!pip install pydicom nibabel scikit-image scipy matplotlib pandas tqdm segmentation-models-pytorch pylibjpeg pylibjpeg-openjpeg pylibjpeg-libjpeg -q

# ── JPEG Lossless desteği için gdcm ─────────────────────────

import subprocess, sys

# Yol 1: pip wheel
r = subprocess.run([sys.executable, "-m", "pip", "install", "gdcm", "-q"],
                   capture_output=True, text=True)
if r.returncode != 0:
    # Yol 2: apt
    subprocess.run(["apt-get", "install", "-y", "python3-gdcm"], 
                   capture_output=True)

# ── pydicom'a handler'ları tanıt ────────────────────────────
import pydicom
pydicom.config.pixel_data_handlers = ["gdcm", "pylibjpeg", "numpy"]

# ── Test ────────────────────────────────────────────────────
from pathlib import Path
test = list(Path("/kaggle/input/datasets/crawford/qureai-headct/qct01").rglob("*.dcm"))[:1]
if test:
    try:
        ds = pydicom.dcmread(str(test[0]))
        arr = ds.pixel_array
        print(f"✅ DICOM okuma çalışıyor: {arr.shape}")
    except Exception as e:
        print(f"❌ DICOM hâlâ okunamıyor: {e}")
        print("   Manuel çözüm: Kaggle secrets'a gdcm ekleyin veya")
        print("   notebook'u gdcm kurulu bir ortamda çalıştırın")
else:
    print("⚠️  Test DICOM bulunamadı")

print("✅ Kurulum tamamlandı")

In [44]:
%%capture
!pip install pydicom nibabel scikit-image scipy matplotlib pandas tqdm segmentation-models-pytorch
!conda install -c conda-forge gdcm -y 2>/dev/null || apt-get install -y python3-gdcm 2>/dev/null || pip install gdcm
import pydicom
pydicom.config.pixel_data_handlers = ["gdcm", "pylibjpeg", "numpy"]
print("✅ Kurulum tamamlandı")

## 📥 2. Veri Kaynağı Seçimi

**Üç mod mevcuttur:**
- **kaggle**: Kaggle'da CQ500 dataset'ini kullan (önerilen)
- **synthetic**: Sentetik test verisi (dataset gerekmez)
- **drive**: Google Drive'dan CQ500 yükle

In [45]:
import os
from pathlib import Path

# ─── AYAR: Veri kaynağınızı seçin ────────────────────────────────────────────────────
DATA_SOURCE = "kaggle"   # "kaggle" veya "drive"

# Drive kullanıyorsanız:
DRIVE_CQ500_PATH = "/content/drive/MyDrive/CQ500"   # kendi yolunuzu yazın

# ──────────────────────────────────────────────────────────────────────────────────────
if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    CQ500_ROOT = DRIVE_CQ500_PATH
elif DATA_SOURCE == "kaggle":
    CQ500_ROOT = "/kaggle/input/datasets/crawford/qureai-headct"
    CQ500_ROOT2 = "/kaggle/working"
else:
    raise ValueError(f"Geçersiz DATA_SOURCE: {DATA_SOURCE}. 'kaggle' veya 'drive' olmalı.")

print(f"Mod: {DATA_SOURCE} | Root: {CQ500_ROOT}")

Mod: kaggle | Root: /kaggle/input/datasets/crawford/qureai-headct


## 🔬 3. Pipeline Kodu

### Bileşenler:
1. **SliceSelectionAlgorithm** - SSA (Qi et al. 2013)
2. **IdealMidlineDetector** - IML detection
3. **DeformedMidlineDetector** - DML detection with U-Net integration
4. **MidlineShiftCalculator** - MLS measurement
5. **BasalCisternCalculator** - CR calculation (Toledo et al. 2021)
6. **RotterdamScoreCalculator** - Final score assembly

In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings, logging
from pathlib import Path
from typing import Optional, Tuple, List, Dict

import pydicom
import nibabel as nib
from scipy import ndimage
from skimage.morphology import remove_small_objects, binary_closing, disk
from skimage.measure import label as sk_label, regionprops
from skimage import transform, morphology
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

print("✅ Kütüphaneler yüklendi")

✅ Kütüphaneler yüklendi


In [47]:
# ════════════════════════════════════════════════════════════
# BÖLÜM A: SLICE SEÇİM ALGORİTMASI (SSA) — Qi et al. 2013
#
# Dört adım (makaledeki sırayla):
#   1. Kafatası Tespiti     : T=250 HU eşiği + S=200 küçük bölge doldurma
#   2. Kapalı Kafatası      : F (bağlantılı bileşen sayısı) ≤ 2
#   3. Kafa İçi Alan Tespiti: Kafatası kütle merkezi → intrakraniyal alan
#   4. Dışbükeylik İncelemesi: Λ_Total = intrakraniyal alan / convex hull alanı
#
# Skor = Λ_Total (dışbükeylik oranı)
# En yüksek skorlu TARGET_SLICE_COUNT adet slice seçilir.
# ════════════════════════════════════════════════════════════

from skimage.morphology import convex_hull_image

class SliceSelectionAlgorithm:
    BONE_HU_THRESHOLD      = 250    # Adım 1: kemik eşiği (makale: T=250)
    SMALL_REGION_THRESH    = 200    # Adım 1: küçük boşluk doldurma (S=200)
    SKULL_CLOSE_THRESH     = 2      # Adım 2: F ≤ 2 (kapalı kafatası)
    TARGET_SLICE_COUNT     = 6      # Seçilecek slice sayısı
    MIN_INTRACRANIAL_RATIO = 0.10   # Adım 3: intrakraniyal alan >= görüntünün %10'u

    def select_slices(self, volume: np.ndarray) -> List[int]:
        scored = []
        for i in range(volume.shape[0]):
            score, valid = self._score_slice(volume[i])
            if valid:
                scored.append((i, score))

        if not scored:
            mid = volume.shape[0] // 2
            return list(range(max(0, mid - 3), min(volume.shape[0], mid + 3)))

        scored.sort(key=lambda x: x[1], reverse=True)
        candidates = [s[0] for s in scored]
        return sorted(self._distribute(candidates, self.TARGET_SLICE_COUNT))

    def _score_slice(self, sl: np.ndarray):
        # ── Adım 1: Kafatası Tespiti ─────────────────────────
        # T=250 HU eşiği, S=200 küçük boşluk doldurma
        bone = sl > self.BONE_HU_THRESHOLD
        bone = remove_small_objects(bone, min_size=self.SMALL_REGION_THRESH)
        if not bone.any():
            return 0.0, False

        # ── Adım 2: Kapalı Kafatası İncelemesi ───────────────
        # Bağlantılı bileşen sayısı F ≤ 2 olmalı
        lbl = sk_label(bone)
        F   = lbl.max()
        if F > self.SKULL_CLOSE_THRESH:
            return 0.0, False

        # ── Adım 3: Kafa İçi Alan Tespiti ────────────────────
        # Kafatası kütle merkezi → binary_fill_holes → intrakraniyal maske
        skull_regions = regionprops(lbl)
        if not skull_regions:
            return 0.0, False

        # Kafatası içini doldur, kemik halkasını çıkar
        intracranial      = ndimage.binary_fill_holes(bone) & ~bone
        intracranial_area = intracranial.sum()

        if intracranial_area / sl.size < self.MIN_INTRACRANIAL_RATIO:
            return 0.0, False

        # ── Adım 4: Dışbükeylik İncelemesi ───────────────────
        # Λ_Total = intrakraniyal alan / convex hull alanı
        # Lateral ventrikül seviyesinde beyin en dışbükey → Λ maksimum
        # Serebellum/beyin sapı içbükeydir → Λ düşük → otomatik elenir
        try:
            hull      = convex_hull_image(intracranial)
            hull_area = hull.sum()
            if hull_area == 0:
                return 0.0, False
            lambda_total = intracranial_area / hull_area
        except Exception:
            return 0.0, False

        return float(lambda_total), True

    def _distribute(self, cands, target):
        if len(cands) <= target:
            return cands
        step = len(cands) / target
        return [cands[int(i * step)] for i in range(target)]

print("✅ SSA (Qi et al. 2013 — 4 adım) hazır")

✅ SSA (Qi et al. 2013 — 4 adım) hazır


In [48]:
# ════════════════════════════════════════════════════════════
# BÖLÜM B: İDEAL MİDLİNE TESPİTİ (IML) — Qi et al. 2013
# ════════════════════════════════════════════════════════════

class IdealMidlineDetector:

    def detect(self, sl: np.ndarray) -> Tuple[float, float]:
        skull = self._skull_mask(sl)
        if not skull.any():
            return sl.shape[1] / 2.0, 0.0
        return self._exhaustive_search(skull)

    def _skull_mask(self, sl):
        bone = remove_small_objects(sl > 400, min_size=200)
        if not bone.any():
            return bone
        lbl = sk_label(bone)
        regions = regionprops(lbl)
        if not regions:
            return bone
        best = max(regions, key=lambda r: r.area)
        return lbl == best.label

    def _exhaustive_search(self, skull, x_range=50, ang_range=10, ang_steps=21):
        H, W   = skull.shape
        cx     = W // 2
        xs     = range(max(W//4, cx - x_range), min(3*W//4, cx + x_range))
        angles = np.linspace(-ang_range, ang_range, ang_steps)
        best_score, best_x, best_ang = -1, float(cx), 0.0
        skull_f = skull.astype(np.float32)

        for ang in angles:
            rot = transform.rotate(skull_f, ang, center=(H//2, cx),
                                   preserve_range=True) > 0.5
            for x in xs:
                s = self._sym_score(rot, x)
                if s > best_score:
                    best_score, best_x, best_ang = s, x, ang

        return best_x, best_ang

    def _sym_score(self, mask, x):
        H, W = mask.shape
        left  = mask[:, max(0, 2*x - W):x]
        right = mask[:, x:min(W, 2*x)]
        right_flip = np.fliplr(right)

        w = min(left.shape[1], right_flip.shape[1])
        if w == 0:
            return 0.0

        L = left[:, -w:]
        R = right_flip[:, :w]

        intersection = (L & R).sum()
        union = (L | R).sum()

        return intersection / max(union, 1)

    def estimate_from_volume(self, vol, slices):
        iml_xs = []
        for idx in slices:
            x, _ = self.detect(vol[idx])
            iml_xs.append(x)
        return np.median(iml_xs), 0.0

In [49]:
# ════════════════════════════════════════════════════════════
# BÖLÜM C: GERÇEK MİDLİNE TESPİTİ (DML) — Qi et al. 2013
#
# Yöntem: Septum pellucidum bazlı çok noktalı DML
#   Her slice'ta sol+sağ lateral ventrikül merkezi tespit edilir.
#   Bu noktalar birleştirilince eğri DML çizgisi oluşur.
#   MLS = eğri DML'nin IML'den maksimum sapması.
#
# Ventrikül filtresi:
#   - Boyut: 1000–30000 px (sulkusları ve artefaktları eler)
#   - IML'ye yakınlık: width × 0.15 (kafatası kenarındaki yapıları eler)
# ════════════════════════════════════════════════════════════

class DeformedMidlineDetector:
    CSF_HU_MIN         = -10
    CSF_HU_MAX         = 15
    MIN_VENTRICLE_AREA = 1000    # Sulkusları ele
    MAX_VENTRICLE_AREA = 30000   # Artefaktları ele
    MAX_IML_DIST_RATIO = 0.15    # IML'den max uzaklık (görüntü genişliğinin %15'i)

    def detect(self, sl, iml_x, pixel_spacing, unet_mask=None):
        brain = self._brain_region(sl)
        if unet_mask is not None:
            brain = brain & ~unet_mask.astype(bool)

        csf    = self._csf(sl, brain)
        dml_x  = self._from_csf(csf, sl.shape[1], iml_x)
        method = "septum_pellucidum"

        if dml_x is None:
            dml_x  = self._centroid_fallback(sl, brain)
            method = "centroid"

        return dml_x, {"method": method, "csf_px": int(csf.sum())}

    def detect_curve(self, volume, slices, iml_x, pixel_spacing, unet_vol=None):
        """
        Birden fazla slice üzerinde DML eğrisini hesapla.
        Her slice için bir (slice_idx, dml_x) noktası döndürür.
        """
        curve_points = []
        for idx in slices:
            sl   = volume[idx]
            mask = unet_vol[idx] if unet_vol is not None else None
            dml_x, dbg = self.detect(sl, iml_x, pixel_spacing, mask)
            if dml_x is not None:
                curve_points.append({
                    "slice_idx": idx,
                    "y":         sl.shape[0] // 2,
                    "dml_x":     dml_x,
                    "debug":     dbg,
                })
        return curve_points

    def _brain_region(self, sl):
        tissue = (sl > -10) & (sl < 150) & ~(sl > 400)
        tissue = binary_closing(tissue, disk(3))
        lbl    = sk_label(tissue)
        regs   = regionprops(lbl)
        if not regs:
            return tissue
        brain  = lbl == max(regs, key=lambda r: r.area).label
        return ndimage.binary_fill_holes(brain)

    def _csf(self, sl, brain):
        csf = (sl >= self.CSF_HU_MIN) & (sl <= self.CSF_HU_MAX) & brain
        return remove_small_objects(csf, min_size=100)

    def _from_csf(self, csf, width, iml_x):
        """
        Septum pellucidum yöntemi:
        1. Ventrikül boyutundaki CSF bölgelerini filtrele
        2. IML'ye yakın olanları tut (kafatası kenarındaki sulkusları ele)
        3. Sol + sağ en büyük bölgenin ortası → DML
        4. Sonuç makul değilse None döndür → centroid fallback
        """
        if not csf.any():
            return None

        regs = regionprops(sk_label(csf))

        # 1. Boyut filtresi
        regs = [r for r in regs
                if self.MIN_VENTRICLE_AREA <= r.area <= self.MAX_VENTRICLE_AREA]
        if not regs:
            return None

        # 2. IML'ye yakınlık filtresi
        max_dist = width * self.MAX_IML_DIST_RATIO
        regs = [r for r in regs if abs(r.centroid[1] - iml_x) <= max_dist]
        if not regs:
            return None

        # 3. Sol/sağ ayır
        left_regs  = [r for r in regs if r.centroid[1] < iml_x]
        right_regs = [r for r in regs if r.centroid[1] >= iml_x]

        if not left_regs or not right_regs:
            if len(regs) >= 2:
                top2  = sorted(regs, key=lambda r: r.area, reverse=True)[:2]
                xs    = sorted([r.centroid[1] for r in top2])
                dml_x = (xs[0] + xs[1]) / 2
            elif len(regs) == 1:
                dml_x = regs[0].centroid[1]
            else:
                return None
        else:
            L     = max(left_regs,  key=lambda r: r.area)
            R     = max(right_regs, key=lambda r: r.area)
            dml_x = (L.centroid[1] + R.centroid[1]) / 2

        # 4. Son makul kontrol
        if abs(dml_x - iml_x) > max_dist:
            return None

        return dml_x

    def _centroid_fallback(self, sl, brain):
        if not brain.any():
            return sl.shape[1] / 2.0
        y, x = ndimage.center_of_mass(brain)
        return x

print("✅ DML (Qi et al. 2013 — Septum Pellucidum + Eğri) hazır")

✅ DML (Qi et al. 2013 — Septum Pellucidum + Eğri) hazır


In [50]:
# ════════════════════════════════════════════════════════════
# BÖLÜM D: MLS HESAPLAMA — Qi et al. 2013
#
# MLS = eğri DML'nin IML'den maksimum sapması
# Aggregation stratejisi:
#   1. Önce septum_pellucidum metoduyla bulunan slice'lar
#      (bunlar gerçek ventrikül tespiti — daha güvenilir)
#   2. Yoksa tüm slice'ların median'ı
#   3. MAD filtresi ile outlier'ları çıkar
# ════════════════════════════════════════════════════════════

class MidlineShiftCalculator:
    ROTTERDAM_THRESHOLD_MM = 5.0

    def per_slice(self, iml_x, dml_x, px_spacing):
        diff_mm = abs(dml_x - iml_x) * px_spacing
        return {
            "mls_mm":              diff_mm,
            "direction":           "right" if dml_x > iml_x else "left",
            "rotterdam_mls_point": int(diff_mm >= self.ROTTERDAM_THRESHOLD_MM),
            "iml_x":               iml_x,
            "dml_x":               dml_x,
        }

    def from_volume(self, volume, slices, iml_x, px_spacing, unet_vol=None):
        """
        Qi et al. 2013 yöntemi:
        1. Her slice'ta DML noktası tespit et → eğri DML
        2. Max MLS = eğri DML'nin IML'den max sapması
        3. Rotterdam için:
           a. Septum pellucidum bulunan slice'ları önceliklendir
           b. MAD filtresi ile outlier slice'ları çıkar
           c. Kalan slice'ların mean'i → final MLS
        """
        dml_det = DeformedMidlineDetector()
        results = []

        for idx in slices:
            mask  = unet_vol[idx] if unet_vol is not None else None
            dml_x, dbg = dml_det.detect(volume[idx], iml_x, px_spacing, mask)
            if dml_x is not None:
                r = self.per_slice(iml_x, dml_x, px_spacing)
                r.update({"slice_idx": idx, "debug": dbg})
                results.append(r)

        if not results:
            return {"mls_mm": 0.0, "rotterdam_mls_point": 0,
                    "slice_results": [], "curve_points": []}

        # Eğri DML noktaları (görselleştirme için)
        curve_points = [
            {"slice_idx": r["slice_idx"], "dml_x": r["dml_x"]}
            for r in results
        ]

        # Max MLS (Qi et al. yöntemi — tüm slice'lar)
        max_mls = max(r["mls_mm"] for r in results)

        # ── Rotterdam için aggregation ────────────────────────
        # Adım 1: Septum pellucidum bulunan slice'ları seç
        # (centroid fallback'ten daha güvenilir)
        sp_results = [r for r in results
                      if r["debug"]["method"] == "septum_pellucidum"]

        if sp_results:
            candidate_values = np.array([r["mls_mm"] for r in sp_results])
        else:
            # Septum bulunamadıysa tüm slice'ları kullan
            candidate_values = np.array([r["mls_mm"] for r in results])

        # Adım 2: MAD filtresi — outlier slice'ları çıkar
        if len(candidate_values) >= 3:
            median    = np.median(candidate_values)
            mad       = np.median(np.abs(candidate_values - median))
            threshold = median + 2 * mad
            filtered  = candidate_values[candidate_values <= threshold]
            mean_mls  = float(np.mean(filtered)) if len(filtered) > 0 else float(median)
        elif len(candidate_values) > 0:
            mean_mls = float(np.mean(candidate_values))
        else:
            mean_mls = 0.0

        return {
            "mls_mm":              round(mean_mls, 2),
            "max_mls_mm":          round(max_mls,  2),
            "rotterdam_mls_point": int(mean_mls >= self.ROTTERDAM_THRESHOLD_MM),
            "slice_results":       results,
            "curve_points":        curve_points,
        }

print("✅ MLS Calculator (Qi et al. 2013 — Septum Öncelikli + MAD) hazır")

✅ MLS Calculator (Qi et al. 2013 — Septum Öncelikli + MAD) hazır


In [51]:
# ════════════════════════════════════════════════════════════
# BÖLÜM E: BASAL CISTERN — Toledo et al. 2021
# ════════════════════════════════════════════════════════════
#
# CR = CSF voksel sayısı / (GM + WM voksel sayısı)
# HU filtresi  : [-5, +45]
# K-means      : k=3, sabit başlangıç [0, 25, 35] HU
# Eşikler      : CR >= 0.20 → normal (0 pt)
#                0.05–0.20  → compressed (1 pt)
#                CR < 0.05  → absent (2 pt)

class BasalCisternCalculator:

    CR_NORMAL_THRESHOLD  = 0.20
    CR_PARTIAL_THRESHOLD = 0.08
    HU_MIN = -5
    HU_MAX =  45
    KMEANS_INIT_CENTERS  = np.array([[0.0], [25.0], [35.0]])
    MIN_VOXELS_FOR_CR    = 50
    MIDBRAIN_SLICE_RANGE = (0.25, 0.42)

    def calculate_cr(self, volume, pixel_spacing=1.0, unet_vol=None, verbose=False):
        midbrain_idx = self._select_midbrain_slice(volume)
        sl = volume[midbrain_idx]

        brain_mask = self._skull_strip(sl)
        if unet_vol is not None:
            brain_mask = brain_mask & ~unet_vol[midbrain_idx].astype(bool)

        hu_filtered = (sl >= self.HU_MIN) & (sl <= self.HU_MAX) & brain_mask
        n_filtered  = hu_filtered.sum()

        if n_filtered < self.MIN_VOXELS_FOR_CR:
            return self._build_result(0.0, midbrain_idx, 0, 0, [], {}, False)

        hu_vals  = sl[hu_filtered].reshape(-1, 1)
        km = KMeans(n_clusters=3, init=self.KMEANS_INIT_CENTERS,
                    n_init=1, max_iter=300, random_state=42)
        km_labels = km.fit_predict(hu_vals)
        centers   = km.cluster_centers_.flatten()

        sorted_ids  = np.argsort(centers)   # 0=CSF, 1=GM, 2=WM
        csf_id = sorted_ids[0]
        gm_id  = sorted_ids[1]
        wm_id  = sorted_ids[2]
        label_map = {int(csf_id): "CSF", int(gm_id): "GM", int(wm_id): "WM"}

        n_csf        = int((km_labels == csf_id).sum())
        n_parenchyma = int((km_labels == gm_id).sum() + (km_labels == wm_id).sum())
        cr           = n_csf / n_parenchyma if n_parenchyma > 0 else 0.0

        return self._build_result(cr, midbrain_idx, n_csf, n_parenchyma,
                                  centers.tolist(), label_map, True)

    def _select_midbrain_slice(self, volume):
        n  = volume.shape[0]
        lo = int(n * self.MIDBRAIN_SLICE_RANGE[0])
        hi = int(n * self.MIDBRAIN_SLICE_RANGE[1])
        best_idx, best_score = (lo + hi) // 2, -1.0
        for i in range(lo, hi + 1):
            sl    = volume[i]
            score = float(((sl >= self.HU_MIN) & (sl <= self.HU_MAX) & ~(sl > 400)).sum())
            if score > best_score:
                best_score, best_idx = score, i
        return best_idx

    def _skull_strip(self, sl):
        from skimage.measure import label, regionprops
        bone   = sl > 400
        tissue = (sl > -50) & (sl < 200) & ~bone
        filled = ndimage.binary_fill_holes(tissue)
        lbl    = label(filled)
        regs   = regionprops(lbl)
        if not regs:
            return filled
        return lbl == max(regs, key=lambda r: r.area).label

    def _build_result(self, cr, midbrain_idx, n_csf, n_parenchyma,
                      centers, label_map, valid):
        if not valid:
            basal_score, status = 2, "absent (invalid)"
        elif cr >= self.CR_NORMAL_THRESHOLD:
            basal_score, status = 0, "normal"
        elif cr >= self.CR_PARTIAL_THRESHOLD:
            basal_score, status = 1, "compressed"
        else:
            basal_score, status = 2, "absent"
        return {
            "compression_ratio":   round(cr, 4),
            "basal_score":         basal_score,
            "status":              status,
            "midbrain_slice_idx":  midbrain_idx,
            "n_csf_voxels":        n_csf,
            "n_parenchyma_voxels": n_parenchyma,
            "kmeans_centers_hu":   [round(c, 2) for c in centers],
            "cluster_labels":      {str(k): v for k, v in label_map.items()},
            "valid":               valid,
        }

In [52]:
# ════════════════════════════════════════════════════════════
# BÖLÜM F: ROTTERDAM SCORE HESAPLAYICI
# ════════════════════════════════════════════════════════════

class RotterdamScoreCalculator:
    """Complete Rotterdam CT Score calculation"""
    
    # Mortality rates from Maas et al. (2005)
    _mortality = {2: 0.07, 3: 0.12, 4: 0.27, 5: 0.54, 6: 0.61}

    def calculate(self, mls_mm, basal=0, edh_absent=1, ivh_sah=0, hemorrhage_volume=0):
        """
        Calculate Rotterdam CT Score
        
        Args:
            mls_mm: Midline shift in millimeters
            basal: Basal cistern status (0=normal, 1=compressed, 2=absent)
            edh_absent: Epidural hematoma (0=present, 1=absent)
            ivh_sah: Intraventricular or subarachnoid hemorrhage (0=absent, 1=present)
            hemorrhage_volume: Total hemorrhage volume in pixels (for mass effect)
        
        Returns:
            dict with score, components, and mortality estimate
        """
        # Component 1: Basal cistern status (0, 1, or 2 points)
        basal_pts = basal
        
        # Component 2: Midline shift (0 or 1 point)
        mls_pts = 1 if mls_mm >= 5.0 else 0
        
        # Component 3: EDH (0 or 1 point)
        edh_pts = edh_absent
        
        # Component 4: IVH or SAH (0 or 1 point)
        ivh_sah_pts = ivh_sah
        
        # Component 5: Mass effect from hemorrhage volume
        # If large hemorrhage (>500 pixels), likely mass effect
        mass_effect_pts = 1 if hemorrhage_volume > 500 else 0
        
        # Total score (range: 2-6)
        # Base score is 1, add all components
        total = 1 + basal_pts + mls_pts + edh_pts + ivh_sah_pts
        total = min(6, max(2, total))  # Clamp to 2-6 range
        
        mortality = self._mortality.get(total, 0.5)
        
        return {
            "rotterdam_score": total,
            "components": {
                "basal_cistern": basal_pts,
                "mls": mls_pts,
                "edh_absent": edh_pts,
                "ivh_sah": ivh_sah_pts,
                "mass_effect": mass_effect_pts
            },
            "estimated_mortality": mortality,
            "risk_category": self._risk_category(total)
        }
    
    def _risk_category(self, score):
        if score <= 2:
            return "Low risk"
        elif score == 3:
            return "Moderate risk"
        elif score == 4:
            return "High risk"
        else:
            return "Very high risk"

## 🤖 4. U-Net Hemorrhage Segmentation

Load pre-trained U-Net model for hemorrhage detection.

In [53]:
import torch
import segmentation_models_pytorch as smp

# Load U-Net model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = smp.Unet(encoder_name="resnet34", in_channels=3, classes=1).to(device)

# Load weights (adjust path for your environment)
try:
    model_path = "/kaggle/input/datasets/rembengbal/unet-model2/unet_ct_ich_best.pth"
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("✅ U-Net model loaded successfully")
except FileNotFoundError:
    print("⚠️  U-Net weights not found - using untrained model (for testing only)")

model.eval()

def predict_volume(volume, model, device):
    """Generate hemorrhage segmentation for entire CT volume"""
    unet_vol = np.zeros_like(volume, dtype=np.uint8)

    def window(im, center, width):
        mn, mx = center - width//2, center + width//2
        return (np.clip(im, mn, mx) - mn) / (mx - mn)

    with torch.no_grad():
        for i in range(volume.shape[0]):
            img_slice = volume[i]
            
            # Three-channel windowing for U-Net input
            c1 = window(img_slice, 40, 80)     # Brain window
            c2 = window(img_slice, 500, 3000)  # Bone window
            c3 = window(img_slice, 175, 50)    # Subdural window
            
            img_stack = np.stack([c1, c2, c3], axis=0).astype(np.float32)
            tensor_img = torch.tensor(img_stack).unsqueeze(0).to(device)

            # Model prediction
            pred = model(tensor_img)
            pred_mask = (torch.sigmoid(pred) > 0.5).float().cpu().numpy()[0, 0]
            unet_vol[i] = pred_mask

    return unet_vol

print("✅ U-Net inference function ready")

Using device: cpu
✅ U-Net model loaded successfully
✅ U-Net inference function ready


In [54]:
def classify_hemorrhage_by_rules(vol, unet_mask):
    """
    HU yoğunluğu + anatomik lokasyon tabanlı kural sistemi.
    Referans: Wintermark et al. (2008), Lee et al. (2020)
    EDH    : periferik bölge (%15) + 55-90 HU
    SAH/IVH: santral bölge  (%25) + 40-70 HU
    vol      : (H, W, D) slice-last
    unet_mask: (H, W, D) slice-last binary maske
    """
    H, W, D  = vol.shape
    center_x = W // 2
    center_y = H // 2
    edh_pixels = sah_ivh_pixels = 0

    for d in range(D):
        sl      = vol[:, :, d]
        mask_sl = unet_mask[:, :, d]
        if mask_sl.sum() == 0:
            continue
        for (y, x) in np.argwhere(mask_sl > 0):
            hu = float(sl[y, x])
            dist_from_edge   = min(y, H - y, x, W - x)
            is_peripheral    = dist_from_edge < H * 0.15
            dist_from_center = np.sqrt((x - center_x)**2 + (y - center_y)**2)
            is_central       = dist_from_center < min(H, W) * 0.25
            if is_peripheral and 55 <= hu <= 90:
                edh_pixels += 1
            elif is_central and 40 <= hu <= 70:
                sah_ivh_pixels += 1

    return edh_pixels >= 200, sah_ivh_pixels >= 100

## 📂 5. DICOM Data Loader

In [55]:
def find_best_series(patient_dir):
    """
    CQ500'de birden fazla seri var:
      - CT Plain        : 5mm, ~30 slice  → DOĞRU seri (standart klinik)
      - CT PLAIN THIN   : 0.625mm, ~240 slice → ince kesit, kullanma
      - CT 4cc sec ...  : kontrast seriler → kullanma
    
    Seçim önceliği:
      1. SliceThickness >= 4mm olan seri
      2. Bulunamazsa en az slice içeren seri (ince kesitlerden kaçın)
    """
    patient_dir = Path(patient_dir)
    all_dcm = list(patient_dir.rglob("*.dcm"))
    if not all_dcm:
        raise ValueError(f"No DICOM files found in {patient_dir}")

    from collections import defaultdict
    series_map = defaultdict(list)
    for f in all_dcm:
        series_map[f.parent].append(f)

    # Her serinin slice thickness'ını oku
    series_info = []
    for series_dir, files in series_map.items():
        try:
            ds        = pydicom.dcmread(str(files[0]))
            thickness = float(getattr(ds, "SliceThickness", 0))
        except Exception:
            thickness = 0.0
        series_info.append((series_dir, files, thickness))

    # 1. Önce thickness >= 4mm olan seriyi seç
    thick_series = [(d, f, t) for d, f, t in series_info if t >= 4.0]
    if thick_series:
        # Birden fazlaysa en fazla slice içereni al
        best = max(thick_series, key=lambda x: len(x[1]))
        return best[1]

    # 2. Yoksa en az slice içereni al (ince kesitlerden kaçın)
    best = min(series_info, key=lambda x: len(x[1]))
    return best[1]


def load_dicom_volume(patient_dir):
    """
    Load DICOM series as 3D volume (HU).
    CQ500 klasör yapısı:
        patient_dir/Unknown Study/CT SeriesAdı/*.dcm
    Birden fazla seri varsa en fazla slice içereni kullanır.
    """
    dcm_files = find_best_series(patient_dir)
    
    slices = []
    for f in dcm_files:
        try:
            slices.append(pydicom.dcmread(str(f)))
        except Exception:
            continue
    
    if not slices:
        raise ValueError(f"Okunabilir DICOM yok: {patient_dir}")
    
    # Z pozisyonuna göre sırala
    def z_pos(s):
        try:
            return float(s.ImagePositionPatient[2])
        except Exception:
            return 0.0
    slices.sort(key=z_pos)
    
    # Pixel spacing
    try:
        pixel_spacing = float(slices[0].PixelSpacing[0])
    except Exception:
        pixel_spacing = 1.0
    
    # 3D volume
    arrays = []
    for s in slices:
        try:
            # force_convert: compressed transfer syntax için pylibjpeg devreye girer
            arr = s.pixel_array.astype(np.float32)
            slope     = float(getattr(s, "RescaleSlope",     1))
            intercept = float(getattr(s, "RescaleIntercept", 0))
            arrays.append(arr * slope + intercept)
        except Exception as pixel_err:
            # İlk slice hata verirse transfer syntax'ı logla
            ts = getattr(s, "file_meta", None)
            ts = getattr(ts, "TransferSyntaxUID", "unknown") if ts else "unknown"
            print(f"    ⚠️  Slice okunamadı (TransferSyntax={ts}): {pixel_err}")
            continue
    
    if not arrays:
        # Hiç slice okunamadıysa transfer syntax bilgisini ver
        sample = slices[0]
        ts = getattr(getattr(sample, "file_meta", None), "TransferSyntaxUID", "bilinmiyor")
        raise ValueError(
            f"Pixel array okunamadı: {patient_dir}\n"
            f"  TransferSyntaxUID: {ts}\n"
            f"  Çözüm: !pip install pylibjpeg pylibjpeg-openjpeg"
        )
    
    volume = np.stack(arrays)
    return volume, pixel_spacing

print("✅ DICOM loader ready")

✅ DICOM loader ready


## 🚀 6. Complete Rotterdam Score Pipeline

Process all patients and calculate Rotterdam scores.

In [56]:
# ════════════════════════════════════════════════════════════
# CELL —  ROTTERDAM PIPELINE + ACCURACY
# ════════════════════════════════════════════════════════════
ICH_PIXEL_THRESHOLD = 3000

def find_all_patient_dirs(root):
    """5mm CT Plain serisi olan hasta klasörlerini bul"""
    from collections import defaultdict
    root = Path(root)
    dirs = []
    for sub in sorted(root.iterdir()):
        if not sub.is_dir():
            continue
        for d in sorted(sub.iterdir()):
            if not (d.is_dir() and d.name.startswith("CQ500")):
                continue
            all_dcm = list(d.rglob("*.dcm"))
            if not all_dcm:
                continue
            series_map = defaultdict(list)
            for f in all_dcm:
                series_map[f.parent].append(f)
            has_5mm = False
            for series_dir, files in series_map.items():
                try:
                    ds = pydicom.dcmread(str(files[0]))
                    thickness = float(getattr(ds, "SliceThickness", 0))
                    if thickness >= 4.0:
                        has_5mm = True
                        break
                except Exception:
                    continue
            if has_5mm:
                dirs.append(d)
    return sorted(dirs)

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

# ── 1. Ground truth yükle ────────────────────────────────────
reads = pd.read_csv("/kaggle/input/datasets/crawford/qureai-headct/reads.csv")

def majority(row, col):
    vals = [row[f"R1:{col}"], row[f"R2:{col}"], row[f"R3:{col}"]]
    vals = [v for v in vals if pd.notna(v)]
    return int(sum(vals) >= 2) if vals else np.nan

reads["gt_MidlineShift"] = reads.apply(lambda r: majority(r, "MidlineShift"), axis=1)
reads["gt_ICH"]          = reads.apply(lambda r: majority(r, "ICH"),          axis=1)
reads["gt_MassEffect"]   = reads.apply(lambda r: majority(r, "MassEffect"),   axis=1)
reads["gt_EDH"]          = reads.apply(lambda r: majority(r, "EDH"),          axis=1)
reads["gt_IVH"]          = reads.apply(lambda r: majority(r, "IVH"),          axis=1)
reads["gt_SAH"]          = reads.apply(lambda r: majority(r, "SAH"),          axis=1)

def name_to_pid(name):
    num   = name.replace("CQ500-CT-", "")
    short = f"CQ500CT{num}"
    return f"{short} {short}"

reads["pid"] = reads["name"].apply(name_to_pid)

# ── 2.  hasta seç ──────────────────────────────────────────
all_dirs = find_all_patient_dirs(CQ500_ROOT)
patient_dirs_ = all_dirs[:300]
print(f"Toplam kullanılabilir: {len(all_dirs)} | İşlenecek: {len(patient_dirs_)}")

# ── 3. Pipeline ──────────────────────────────────────────────
ssa_        = SliceSelectionAlgorithm()
iml_det_    = IdealMidlineDetector()
mls_calc_   = MidlineShiftCalculator()
basal_calc_ = BasalCisternCalculator()
rot_calc_   = RotterdamScoreCalculator()

results_ = []
errors_  = []

for i, patient_dir in enumerate(patient_dirs_):
    pid = patient_dir.name
    print(f"[{i+1:2d}/300] {pid}", end=" ... ")

    try:
        vol, px_sp        = load_dicom_volume(patient_dir)
        unet              = predict_volume(vol, model, device)
    
        # (D,H,W) → (H,W,D) dönüşümü
        vol_hwd  = np.transpose(vol,  (1, 2, 0))
        unet_hwd = np.transpose(unet, (1, 2, 0))
    
        sel_slices        = ssa_.select_slices(vol)
        iml_x, _          = iml_det_.estimate_from_volume(vol, sel_slices)
        mls_result        = mls_calc_.from_volume(vol, sel_slices, iml_x, px_sp, unet)
        basal_result      = basal_calc_.calculate_cr(vol, px_sp, unet)
    
        mls_mm            = mls_result["mls_mm"]
        cr                = basal_result["compression_ratio"]
        basal_score       = basal_result["basal_score"]
        hemorrhage_volume = unet.sum()
    
        # ── Kural tabanlı EDH/SAH/IVH tespiti ───────────────
        edh_present, sah_ivh_present = classify_hemorrhage_by_rules(vol_hwd, unet_hwd)
        edh_absent = 0 if edh_present else 1
        ivh_sah    = 1 if sah_ivh_present else 0
    
        rotterdam = rot_calc_.calculate(
            mls_mm=mls_mm, basal=basal_score,
            edh_absent=edh_absent, ivh_sah=ivh_sah,
            hemorrhage_volume=hemorrhage_volume
        )
        results_.append({
            "pid"              : pid,
            "mls_mm"           : round(mls_mm, 1),
            "pred_mls_pos"     : int(mls_mm >= 5.0),
            "basal_cr"         : round(cr, 4),
            "basal_score"      : basal_score,
            "basal_status"     : basal_result["status"],
            "unet_blood_px"    : int(hemorrhage_volume),
            "pred_ich"         : int(hemorrhage_volume > ICH_PIXEL_THRESHOLD),
            "pred_edh"         : int(edh_present),
            "pred_sah_ivh"     : int(sah_ivh_present),
            "pred_basal_abnormal": int(basal_score > 0),
            "rotterdam_score"  : rotterdam["rotterdam_score"],
            "mortality"        : round(rotterdam["estimated_mortality"], 2),
            "risk"             : rotterdam["risk_category"],
        })

        print(
            f"MLS={mls_mm:.1f}mm  CR={cr:.3f}  "
            f"EDH={'✅' if edh_present else '—'}  "
            f"SAH/IVH={'✅' if sah_ivh_present else '—'}  "
            f"Rotterdam={rotterdam['rotterdam_score']}"
        )

    except Exception as e:
        errors_.append({"pid": pid, "error": str(e)})
        print(f"❌ {e}")
        continue

print(f"\n✅ Tamamlandı: {len(results_)} hasta | ❌ Hata: {len(errors_)} hasta")

# ── 4. Ground truth ile birleştir ────────────────────────────
df_ = pd.DataFrame(results_)
merged_ = df_.merge(
    reads[["pid", "gt_MidlineShift", "gt_ICH", "gt_MassEffect",
           "gt_EDH", "gt_IVH", "gt_SAH"]],
    on="pid", how="left"
)
merged_["gt_basal_abnormal"] = merged_["gt_MassEffect"]
merged_["gt_sah_ivh"] = (
    (merged_["gt_SAH"] == 1) | (merged_["gt_IVH"] == 1)
).astype(int)

matched = merged_["gt_MidlineShift"].notna().sum()
print(f"\nGround truth eşleşen: {matched} / {len(merged_)} hasta")

# ── 5. Metrik fonksiyonu ─────────────────────────────────────
def print_metrics(y_true, y_pred, title):
    valid = [(t, p) for t, p in zip(y_true, y_pred) if pd.notna(t)]
    if not valid:
        print(f"  {title}: ground truth yok\n")
        return
    yt  = [int(v[0]) for v in valid]
    yp  = [int(v[1]) for v in valid]
    n   = len(yt)
    cm  = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0,0], 0, 0, 0)
    acc  = accuracy_score(yt, yp)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    try:
        auc = roc_auc_score(yt, yp)
    except Exception:
        auc = float("nan")
    correct = sum(t == p for t, p in zip(yt, yp))
    print(f"  Accuracy       : {acc:.1%}  ({correct}/{n})")
    print(f"  Sensitivity    : {sens:.1%}  (TP={tp}, FN={fn})")
    print(f"  Specificity    : {spec:.1%}  (TN={tn}, FP={fp})")
    print(f"  PPV (Precision): {ppv:.1%}")
    print(f"  NPV            : {npv:.1%}")
    print(f"  AUC-ROC        : {auc:.3f}")
    print(f"  Confusion Matrix:")
    print(f"              Pred=0  Pred=1")
    print(f"    True=0    {tn:5d}   {fp:5d}")
    print(f"    True=1    {fn:5d}   {tp:5d}")

# ── 6. Accuracy sonuçları ────────────────────────────────────
print("\n" + "="*60)
print("① MLS TESPİTİ")
print("="*60)
print_metrics(merged_["gt_MidlineShift"], merged_["pred_mls_pos"], "MLS")

print("\n" + "="*60)
print("② ICH TESPİTİ (Binary U-Net)")
print("="*60)
print_metrics(merged_["gt_ICH"], merged_["pred_ich"], "ICH")

print("\n" + "="*60)
print("③ EDH TESPİTİ (Kural Sistemi)")
print("="*60)
print_metrics(merged_["gt_EDH"], merged_["pred_edh"], "EDH")

print("\n" + "="*60)
print("④ SAH/IVH TESPİTİ (Kural Sistemi)")
print("="*60)
print_metrics(merged_["gt_sah_ivh"], merged_["pred_sah_ivh"], "SAH/IVH")

print("\n" + "="*60)
print("⑤ BASAL SİSTERN (MassEffect proxy)")
print("="*60)
print_metrics(merged_["gt_basal_abnormal"], merged_["pred_basal_abnormal"], "Basal")

print("\n" + "="*60)
print("⑥ ROTTERDAM SKORU DAĞILIMI")
print("="*60)
score_dist = merged_["rotterdam_score"].value_counts().sort_index()
for score, count in score_dist.items():
    bar = "█" * count
    print(f"  Rotterdam {score}: {bar} ({count} hasta)")

print("\n" + "="*60)
print("ÖZET")
print("="*60)
for label, yt, yp in [
    ("MLS    ", merged_["gt_MidlineShift"],    merged_["pred_mls_pos"]),
    ("ICH    ", merged_["gt_ICH"],             merged_["pred_ich"]),
    ("EDH    ", merged_["gt_EDH"],             merged_["pred_edh"]),
    ("SAH/IVH", merged_["gt_sah_ivh"],         merged_["pred_sah_ivh"]),
    ("Basal* ", merged_["gt_basal_abnormal"],  merged_["pred_basal_abnormal"]),
]:
    valid = [(t, p) for t, p in zip(yt, yp) if pd.notna(t)]
    if not valid:
        continue
    yt_ = [int(v[0]) for v in valid]
    yp_ = [int(v[1]) for v in valid]
    acc = accuracy_score(yt_, yp_)
    correct = sum(t == p for t, p in zip(yt_, yp_))
    print(f"  {label}: {acc:.1%}  ({correct}/{len(yt_)})")
print("  * Basal için MassEffect proxy kullanıldı")

# ── 7. CSV kaydet ────────────────────────────────────────────
out = Path("/kaggle/working") / "rotterdam_scores_300.csv"
df_.to_csv(out, index=False)
print(f"\n💾 Kaydedildi: {out}")

Toplam kullanılabilir: 398 | İşlenecek: 300
[ 1/300] CQ500CT0 CQ500CT0 ... MLS=2.5mm  CR=0.096  EDH=—  SAH/IVH=—  Rotterdam=3
[ 2/300] CQ500CT101 CQ500CT101 ... MLS=12.9mm  CR=0.061  EDH=—  SAH/IVH=—  Rotterdam=5
[ 3/300] CQ500CT102 CQ500CT102 ... MLS=2.5mm  CR=0.051  EDH=—  SAH/IVH=—  Rotterdam=4
[ 4/300] CQ500CT103 CQ500CT103 ... MLS=1.1mm  CR=0.045  EDH=—  SAH/IVH=✅  Rotterdam=5
[ 5/300] CQ500CT104 CQ500CT104 ... MLS=0.1mm  CR=0.045  EDH=—  SAH/IVH=—  Rotterdam=4
[ 6/300] CQ500CT105 CQ500CT105 ... MLS=0.5mm  CR=0.107  EDH=—  SAH/IVH=—  Rotterdam=3
[ 7/300] CQ500CT106 CQ500CT106 ... MLS=9.2mm  CR=0.148  EDH=—  SAH/IVH=✅  Rotterdam=5
[ 8/300] CQ500CT107 CQ500CT107 ... MLS=13.3mm  CR=0.081  EDH=✅  SAH/IVH=✅  Rotterdam=4
[ 9/300] CQ500CT108 CQ500CT108 ... MLS=8.9mm  CR=0.178  EDH=✅  SAH/IVH=✅  Rotterdam=4
[10/300] CQ500CT109 CQ500CT109 ... MLS=8.7mm  CR=0.060  EDH=—  SAH/IVH=✅  Rotterdam=6
[11/300] CQ500CT110 CQ500CT110 ... MLS=0.6mm  CR=0.181  EDH=—  SAH/IVH=—  Rotterdam=3
[12/300] CQ5

In [57]:
# ════════════════════════════════════════════════════════════
# CELL — ROTTERDAM SKORU ACCURACY ANALİZİ
# CQ500 reads.csv ile pipeline tahminlerini karşılaştırır
# ════════════════════════════════════════════════════════════

from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

reads = pd.read_csv("/kaggle/input/datasets/crawford/qureai-headct/reads.csv")

# ── 1. Ground truth ──────────────────────────────────────────
def majority(row, col):
    vals = [row[f"R1:{col}"], row[f"R2:{col}"], row[f"R3:{col}"]]
    vals = [v for v in vals if pd.notna(v)]
    return int(sum(vals) >= 2) if vals else np.nan

reads["gt_MidlineShift"] = reads.apply(lambda r: majority(r, "MidlineShift"), axis=1)
reads["gt_ICH"]          = reads.apply(lambda r: majority(r, "ICH"),          axis=1)
reads["gt_MassEffect"]   = reads.apply(lambda r: majority(r, "MassEffect"),   axis=1)

def name_to_pid(name):
    num = name.replace("CQ500-CT-", "")
    short = f"CQ500CT{num}"
    return f"{short} {short}"

reads["pid"] = reads["name"].apply(name_to_pid)

# ── 2. Pipeline sonuçlarıyla birleştir ───────────────────────
df_pred = pd.DataFrame(results_)
merged  = df_pred.merge(
    reads[["pid", "gt_MidlineShift", "gt_ICH", "gt_MassEffect"]],
    on="pid", how="left"
)
merged["gt_basal_abnormal"] = merged["gt_MassEffect"]
merged["pred_basal_abnormal"] = (merged["basal_score"] > 0).astype(int)

print(f"Eşleşen hasta: {merged['gt_MidlineShift'].notna().sum()} / {len(merged)}")

# ── 3. Metrik fonksiyonu ─────────────────────────────────────
def print_metrics(y_true, y_pred, title):
    valid = [(t, p) for t, p in zip(y_true, y_pred) if pd.notna(t)]
    if not valid:
        print(f"  {title}: ground truth yok\n")
        return
    yt  = [int(v[0]) for v in valid]
    yp  = [int(v[1]) for v in valid]
    n   = len(yt)
    cm  = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0,0], 0, 0, 0)
    acc  = accuracy_score(yt, yp)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    try:    auc = roc_auc_score(yt, yp)
    except: auc = float("nan")
    correct = sum(t == p for t, p in zip(yt, yp))
    print(f"  Accuracy       : {acc:.1%}  ({correct}/{n})")
    print(f"  Sensitivity    : {sens:.1%}  (TP={tp}, FN={fn})")
    print(f"  Specificity    : {spec:.1%}  (TN={tn}, FP={fp})")
    print(f"  PPV (Precision): {ppv:.1%}")
    print(f"  NPV            : {npv:.1%}")
    print(f"  AUC-ROC        : {auc:.3f}")
    print(f"  Confusion Matrix:")
    print(f"              Pred=0  Pred=1")
    print(f"    True=0    {tn:5d}   {fp:5d}")
    print(f"    True=1    {fn:5d}   {tp:5d}")

# ── 4. Bileşen bazlı accuracy ────────────────────────────────
print("\n" + "="*60)
print("① MLS TESPİTİ")
print("="*60)
print_metrics(merged["gt_MidlineShift"], merged["pred_mls_pos"], "MLS")

print("\n" + "="*60)
print("② ICH TESPİTİ")
print("="*60)
print_metrics(merged["gt_ICH"], merged["pred_ich"], "ICH")

print("\n" + "="*60)
print("③ BASAL SİSTERN (MassEffect proxy)")
print("   ⚠️  CQ500'de basal cistern etiketi yok")
print("="*60)
print_metrics(merged["gt_basal_abnormal"], merged["pred_basal_abnormal"], "Basal")

print("\n" + "="*60)
print("④ ROTTERDAM SKORU DAĞILIMI")
print("="*60)
score_dist = df_pred["rotterdam_score"].value_counts().sort_index()
for score, count in score_dist.items():
    print(f"  Rotterdam {score}: {'█'*count} ({count} hasta)")

print("\n" + "="*60)
print("ÖZET")
print("="*60)
for label, yt, yp in [
    ("MLS    ", merged["gt_MidlineShift"],    merged["pred_mls_pos"]),
    ("ICH    ", merged["gt_ICH"],             merged["pred_ich"]),
    ("Basal* ", merged["gt_basal_abnormal"],  merged["pred_basal_abnormal"]),
]:
    valid = [(t, p) for t, p in zip(yt, yp) if pd.notna(t)]
    if not valid: continue
    yt_ = [int(v[0]) for v in valid]
    yp_ = [int(v[1]) for v in valid]
    acc = accuracy_score(yt_, yp_)
    correct = sum(t==p for t,p in zip(yt_,yp_))
    print(f"  {label}: {acc:.1%}  ({correct}/{len(yt_)})")
print("  * Basal için MassEffect proxy kullanıldı")

# ── 5. Hasta bazlı detay ─────────────────────────────────────
print("\n" + "="*60)
print("HASTA BAZLI DETAY")
print("="*60)
print(f"{'Hasta':<14} {'MLS_mm':>6} {'MLS_P':>5} {'MLS_G':>5} "
      f"{'ICH_P':>5} {'ICH_G':>5} {'Bas_P':>5} {'Bas_G':>5} {'Score':>5}")
print("─" * 65)
for _, row in merged.iterrows():
    pid    = row["pid"].split()[0]
    gt_mls = int(row["gt_MidlineShift"])    if pd.notna(row["gt_MidlineShift"])   else "?"
    gt_ich = int(row["gt_ICH"])             if pd.notna(row["gt_ICH"])            else "?"
    gt_bas = int(row["gt_basal_abnormal"])  if pd.notna(row["gt_basal_abnormal"]) else "?"
    def mark(pred, gt):
        if gt == "?": return str(pred)
        return f"{pred}{'✅' if pred==int(gt) else '❌'}"
    print(f"{pid:<14} {row['mls_mm']:>6.1f} "
          f"{mark(int(row['pred_mls_pos']),       gt_mls):>7} {str(gt_mls):>5} "
          f"{mark(int(row['pred_ich']),           gt_ich):>7} {str(gt_ich):>5} "
          f"{mark(int(row['pred_basal_abnormal']),gt_bas):>7} {str(gt_bas):>5} "
          f"{int(row['rotterdam_score']):>5}")

Eşleşen hasta: 300 / 300

① MLS TESPİTİ
  Accuracy       : 77.3%  (232/300)
  Sensitivity    : 69.7%  (TP=23, FN=10)
  Specificity    : 78.3%  (TN=209, FP=58)
  PPV (Precision): 28.4%
  NPV            : 95.4%
  AUC-ROC        : 0.740
  Confusion Matrix:
              Pred=0  Pred=1
    True=0      209      58
    True=1       10      23

② ICH TESPİTİ
  Accuracy       : 71.7%  (215/300)
  Sensitivity    : 51.9%  (TP=70, FN=65)
  Specificity    : 87.9%  (TN=145, FP=20)
  PPV (Precision): 77.8%
  NPV            : 69.0%
  AUC-ROC        : 0.699
  Confusion Matrix:
              Pred=0  Pred=1
    True=0      145      20
    True=1       65      70

③ BASAL SİSTERN (MassEffect proxy)
   ⚠️  CQ500'de basal cistern etiketi yok
  Accuracy       : 35.0%  (105/300)
  Sensitivity    : 84.4%  (TP=65, FN=12)
  Specificity    : 17.9%  (TN=40, FP=183)
  PPV (Precision): 26.2%
  NPV            : 76.9%
  AUC-ROC        : 0.512
  Confusion Matrix:
              Pred=0  Pred=1
    True=0       40     18

In [58]:
# ════════════════════════════════════════════════════════════
# — ROTTERDAM SCORE PIPELINE

# ════════════════════════════════════════════════════════════

ICH_PIXEL_THRESHOLD = 3000    # False positive azaltmak için yükseltildi 

if True:
    print("🏥 ROTTERDAM SCORE CALCULATION\n" + "="*60)

    ssa       = SliceSelectionAlgorithm()
    iml_det   = IdealMidlineDetector()
    mls_calc  = MidlineShiftCalculator()
    basal_calc= BasalCisternCalculator()
    rot_calc  = RotterdamScoreCalculator()

    results     = []
    # ── CQ500 klasör yapısı ──────────────────────────────────────────────────
    # Dataset içindeki gerçek patient dizinlerini bul.
    # Kaggle'da CQ500 şu iç içe yapıdadır:
    #   /kaggle/input/.../qureai-headct/
    #       qct01/CQ500CT0 CQ500CT0/
    #       qct02/CQ500CT1 CQ500CT1/  ...vb.
    # Bunun yanı sıra semlink kurulduysa doğrudan CQ500_ROOT altında da olabilir.
    # Her iki durumu da ele alıyoruz.

    def find_patient_dirs(root):
        """
        CQ500 yapısı: root/qct01/CQ500CT0 CQ500CT0/
        qct* alt klasörlerinin içindeki CQ500* dizinlerini toplar.
        """
        root = Path(root)
        dirs = []
        for sub in sorted(root.iterdir()):
            if not sub.is_dir():
                continue
            # qct01, qct02 ... gibi alt klasörler
            for d in sorted(sub.iterdir()):
                if d.is_dir() and d.name.startswith("CQ500"):
                    dirs.append(d)
        # Bulunamazsa doğrudan root altına bak
        if not dirs:
            dirs = [d for d in root.iterdir()
                    if d.is_dir() and d.name.startswith("CQ500")]
        return sorted(dirs)

    all_patient_dirs = find_patient_dirs(CQ500_ROOT)
    print(f"  Toplam bulunan hasta klasörü: {len(all_patient_dirs)}")
    if all_patient_dirs:
        print(f"  Örnek path: {all_patient_dirs[0]}")
    else:
        print("  ❌ Hiç CQ500 klasörü bulunamadı!")
        print(f"     Aranan root: {CQ500_ROOT}")
        import os
        print(f"     Root içeriği: {os.listdir(CQ500_ROOT)[:10]}")

    patient_dirs = all_patient_dirs[:20]   # İlk 20 hasta

    for patient_dir in patient_dirs:
        pid = patient_dir.name
        print(f"\n{'─'*60}")
        print(f"Processing: {pid}")

        try:
            # Volüm yükle
            volume, pixel_spacing = load_dicom_volume(patient_dir)
            print(f"  Volume: {volume.shape}, Spacing: {pixel_spacing:.2f} mm")

            # 1. U-Net hemoraji segmentasyonu
            print("  [1/5] Running U-Net segmentation...")
            unet_vol         = predict_volume(volume, model, device)
            hemorrhage_volume = unet_vol.sum()
            print(f"        Hemorrhage pixels: {int(hemorrhage_volume)}")

            # 2. Slice seçimi
            print("  [2/5] Selecting optimal slices...")
            selected_slices = ssa.select_slices(volume)
            print(f"        Selected: {len(selected_slices)} slices")

            # 3. İdeal midline tespiti
            print("  [3/5] Detecting ideal midline...")
            iml_x, _ = iml_det.estimate_from_volume(volume, selected_slices)
            print(f"        IML at x={iml_x:.1f}")

            # 4. MLS hesaplama
            print("  [4/5] Calculating midline shift...")
            mls_result = mls_calc.from_volume(
                volume, selected_slices, iml_x, pixel_spacing, unet_vol
            )
            mls_mm = mls_result["mls_mm"]
            print(f"        MLS: {mls_mm:.1f} mm")

            # 5. Bazal sistern CR (Toledo et al. 2021)
            print("  [5/5] Calculating basal cistern CR...")
            basal_result = basal_calc.calculate_cr(volume, pixel_spacing, unet_vol)
            cr           = basal_result["compression_ratio"]
            basal_score  = basal_result["basal_score"]
            print(f"        CR: {cr:.4f} → {basal_result['status']}")

            # 6. Hemoraji tipi sınıflandırması
            # ICH threshold 5000 px — false positive azaltma
            ivh_sah    = 1 if hemorrhage_volume > ICH_PIXEL_THRESHOLD else 0
            edh_absent = 1   # EDH tespiti için ayrı modül gerekiyor

            # 7. Rotterdam skoru
            rotterdam = rot_calc.calculate(
                mls_mm=mls_mm,
                basal=basal_score,
                edh_absent=edh_absent,
                ivh_sah=ivh_sah,
                hemorrhage_volume=hemorrhage_volume
            )

            print(f"\n  🏆 ROTTERDAM SCORE: {rotterdam['rotterdam_score']}")
            print(f"     Risk: {rotterdam['risk_category']}")
            print(f"     Mortality estimate: {rotterdam['estimated_mortality']*100:.0f}%")

            results.append({
                "pid":             pid,
                "mls_mm":          round(mls_mm, 1),
                "pred_mls_pos":    int(mls_mm >= 5.0),
                "basal_cr":        round(cr, 4),
                "basal_score":     basal_score,
                "basal_status":    basal_result["status"],
                "unet_blood_px":   int(hemorrhage_volume),
                "pred_ich":        ivh_sah,
                "rotterdam_score": rotterdam["rotterdam_score"],
                "mortality":       round(rotterdam["estimated_mortality"], 2),
                "risk":            rotterdam["risk_category"],
                # Pipeline için gerekli ham veriler (görselleştirmede kullanılır)
                "_mls_result":     mls_result,
                "_basal_result":   basal_result,
                "_selected_slices":selected_slices,
                "_iml_x":          iml_x,
            })

        except Exception as e:
            import traceback
            print(f"  ❌ Error: {e}")
            traceback.print_exc()
            continue

    # Özet tablo (ham pipeline verilerini çıkar)
    df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")}
                       for r in results])

    print("\n" + "="*60)
    print("📊 SUMMARY RESULTS")
    print("="*60)
    print(df.to_string(index=False))

    output_path = Path(CQ500_ROOT2) / "rotterdam_scores.csv"
    df.to_csv(output_path, index=False)
    print(f"\n💾 Results saved to: {output_path}")

🏥 ROTTERDAM SCORE CALCULATION
  Toplam bulunan hasta klasörü: 491
  Örnek path: /kaggle/input/datasets/crawford/qureai-headct/qct01/CQ500CT0 CQ500CT0

────────────────────────────────────────────────────────────
Processing: CQ500CT0 CQ500CT0
  Volume: (30, 512, 512), Spacing: 0.45 mm
  [1/5] Running U-Net segmentation...
        Hemorrhage pixels: 4082
  [2/5] Selecting optimal slices...
        Selected: 4 slices
  [3/5] Detecting ideal midline...
        IML at x=258.5
  [4/5] Calculating midline shift...
        MLS: 2.5 mm
  [5/5] Calculating basal cistern CR...
        CR: 0.0959 → compressed

  🏆 ROTTERDAM SCORE: 4
     Risk: High risk
     Mortality estimate: 27%

────────────────────────────────────────────────────────────
Processing: CQ500CT10 CQ500CT10
  Volume: (224, 512, 512), Spacing: 0.46 mm
  [1/5] Running U-Net segmentation...
        Hemorrhage pixels: 15876
  [2/5] Selecting optimal slices...
        Selected: 6 slices
  [3/5] Detecting ideal midline...
        IML at

In [60]:
# ════════════════════════════════════════════════════════════
# CELL  — GROUND TRUTH KARŞILAŞTIRMA & ACCURACY
# ════════════════════════════════════════════════════════════

from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score)

reads = pd.read_csv("/kaggle/input/datasets/crawford/qureai-headct/reads.csv")

# ── 1. Konsensüs ground truth (majority vote: 2/3 radyolog) ─────
def majority(row, col):
    vals = [row[f"R1:{col}"], row[f"R2:{col}"], row[f"R3:{col}"]]
    vals = [v for v in vals if pd.notna(v)]
    return int(sum(vals) >= 2) if vals else np.nan

reads["gt_MidlineShift"] = reads.apply(lambda r: majority(r, "MidlineShift"), axis=1)
reads["gt_ICH"]          = reads.apply(lambda r: majority(r, "ICH"),          axis=1)
reads["gt_IVH"]          = reads.apply(lambda r: majority(r, "IVH"),          axis=1)
reads["gt_SAH"]          = reads.apply(lambda r: majority(r, "SAH"),          axis=1)
reads["gt_EDH"]          = reads.apply(lambda r: majority(r, "EDH"),          axis=1)
reads["gt_MassEffect"]   = reads.apply(lambda r: majority(r, "MassEffect"),   axis=1)

# ── 2. Name → pid eşlemesi ──────────────────────────────────────
def name_to_pid(name):
    num   = name.replace("CQ500-CT-", "")
    short = f"CQ500CT{num}"
    return f"{short} {short}"

reads["pid"] = reads["name"].apply(name_to_pid)

# ── 3. Pipeline sonuçlarıyla birleştir ──────────────────────────
df_pred = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")}
                        for r in results])

merged = df_pred.merge(
    reads[["pid", "gt_MidlineShift", "gt_ICH", "gt_IVH",
           "gt_SAH", "gt_EDH", "gt_MassEffect"]],
    on="pid", how="left"
)

# Basal cistern için ground truth:
# CQ500'de doğrudan "basal cistern" etiketi yok.
# MassEffect (kitle etkisi) en yakın proxy — compress/absent → MassEffect=1
merged["gt_basal_abnormal"] = merged["gt_MassEffect"]  # 0=normal, 1=anormal
merged["pred_basal_abnormal"] = (merged["basal_score"] > 0).astype(int)

# ── 4. Metrik fonksiyonu ────────────────────────────────────────
def print_metrics(y_true, y_pred, title):
    valid = [(t, p) for t, p in zip(y_true, y_pred) if pd.notna(t)]
    if not valid:
        print(f"  {title}: ground truth yok\n")
        return

    yt  = [int(v[0]) for v in valid]
    yp  = [int(v[1]) for v in valid]
    n   = len(yt)
    cm  = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0,0], 0, 0, 0)

    acc  = accuracy_score(yt, yp)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    try:
        auc = roc_auc_score(yt, yp)
    except Exception:
        auc = float("nan")

    correct = sum(t == p for t, p in zip(yt, yp))

    print(f"  Accuracy      : {acc:.1%}  ({correct}/{n})")
    print(f"  Sensitivity   : {sens:.1%}  (TP={tp}, FN={fn})")
    print(f"  Specificity   : {spec:.1%}  (TN={tn}, FP={fp})")
    print(f"  PPV (Precision): {ppv:.1%}")
    print(f"  NPV           : {npv:.1%}")
    print(f"  AUC-ROC       : {auc:.3f}")
    print(f"  Confusion Matrix:")
    print(f"              Pred=0  Pred=1")
    print(f"    True=0    {tn:5d}   {fp:5d}")
    print(f"    True=1    {fn:5d}   {tp:5d}")

# ── 5. Hasta bazlı detay tablosu ────────────────────────────────
print("=" * 65)
print("HASTA BAZLI DETAY")
print("=" * 65)
print(f"{'Hasta':<14} {'MLS_mm':>6} {'MLS_P':>5} {'MLS_G':>5} "
      f"{'ICH_P':>5} {'ICH_G':>5} "
      f"{'Bas_P':>5} {'Bas_G':>5} "
      f"{'Score':>5}")
print("─" * 65)
for _, row in merged.iterrows():
    pid    = row["pid"].split()[0]
    gt_mls = int(row["gt_MidlineShift"]) if pd.notna(row["gt_MidlineShift"]) else "?"
    gt_ich = int(row["gt_ICH"])          if pd.notna(row["gt_ICH"])          else "?"
    gt_bas = int(row["gt_basal_abnormal"]) if pd.notna(row["gt_basal_abnormal"]) else "?"

    def mark(pred, gt):
        if gt == "?": return str(pred)
        return f"{pred}{'✅' if pred==gt else '❌'}"

    print(f"{pid:<14} {row['mls_mm']:>6.1f} "
          f"{mark(int(row['pred_mls_pos']),   gt_mls):>7} {str(gt_mls):>5} "
          f"{mark(int(row['pred_ich']),        gt_ich):>7} {str(gt_ich):>5} "
          f"{mark(int(row['pred_basal_abnormal']), gt_bas):>7} {str(gt_bas):>5} "
          f"{int(row['rotterdam_score']):>5}")

# ── 6. Bileşen bazlı accuracy ───────────────────────────────────
print("\n")
print("=" * 65)
print("① MLS TESPİTİ (pred: ≥5mm=1 | gt: radyolog konsensüsü)")
print("=" * 65)
print_metrics(merged["gt_MidlineShift"], merged["pred_mls_pos"], "MLS")

print("\n")
print("=" * 65)
print("② ICH TESPİTİ (pred: U-Net px>threshold | gt: radyolog konsensüsü)")
print("=" * 65)
print_metrics(merged["gt_ICH"], merged["pred_ich"], "ICH")

print("\n")
print("=" * 65)
print("③ BASAL SİSTERN (pred: CR threshold | gt: MassEffect proxy)")
print("   ⚠️  CQ500'de basal cistern etiketi yok — MassEffect kullanıldı")
print("=" * 65)
print_metrics(merged["gt_basal_abnormal"], merged["pred_basal_abnormal"], "Basal")

print("\n")
print("=" * 65)
print("④ ROTTERDAM SKORU (referans dağılımı)")
print("=" * 65)
print("  Not: CQ500'de Rotterdam ground truth skoru yok.")
print("  Pipeline tahmin dağılımı:")
score_dist = merged["rotterdam_score"].value_counts().sort_index()
for score, count in score_dist.items():
    bar = "█" * count
    print(f"    Rotterdam {score}: {bar} ({count} hasta)")

print("\n")
print("=" * 65)
print("ÖZET")
print("=" * 65)
for label, yt, yp in [
    ("MLS    ", merged["gt_MidlineShift"],    merged["pred_mls_pos"]),
    ("ICH    ", merged["gt_ICH"],             merged["pred_ich"]),
    ("Basal* ", merged["gt_basal_abnormal"],  merged["pred_basal_abnormal"]),
]:
    valid = [(t, p) for t, p in zip(yt, yp) if pd.notna(t)]
    if not valid: continue
    yt_ = [int(v[0]) for v in valid]
    yp_ = [int(v[1]) for v in valid]
    acc = accuracy_score(yt_, yp_)
    correct = sum(t==p for t,p in zip(yt_,yp_))
    print(f"  {label}: {acc:.1%}  ({correct}/{len(yt_)})")
print("  * Basal için MassEffect proxy kullanıldı")

HASTA BAZLI DETAY
Hasta          MLS_mm MLS_P MLS_G ICH_P ICH_G Bas_P Bas_G Score
─────────────────────────────────────────────────────────────────
CQ500CT0          2.5      0✅     0      1❌     0      1❌     0     4
CQ500CT10         1.3      0✅     0      1✅     1      1❌     0     4
CQ500CT100        0.8      0✅     0      0✅     0      1❌     0     3
CQ500CT101       12.9      1❌     0      0✅     0      1❌     0     5
CQ500CT102        2.5      0✅     0      0❌     1      1❌     0     4
CQ500CT103        1.1      0✅     0      0✅     0      1❌     0     4
CQ500CT104        0.1      0✅     0      0✅     0      1❌     0     4
CQ500CT105        0.5      0✅     0      0✅     0      1❌     0     3
CQ500CT106        9.2      1❌     0      0❌     1      1❌     0     4
CQ500CT107       13.3      1✅     1      1✅     1      1✅     1     5
CQ500CT108        8.9      1❌     0      1❌     0      1✅     1     5
CQ500CT109        8.7      1❌     0      1✅     1      1❌     0     6
CQ500CT110  

In [61]:
# ════════════════════════════════════════════════════════════
# CELL —  HASTA: SEÇİLİ 6 SLICE + CSV KAYDET
# ════════════════════════════════════════════════════════════

import shutil
from pathlib import Path
from datetime import datetime

def find_patient_dir_by_pid(pid, root):
    root = Path(root)
    for sub in sorted(root.iterdir()):
        if not sub.is_dir():
            continue
        candidate = sub / pid
        if candidate.is_dir():
            return candidate
    return None

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
save_root = Path(f"/kaggle/working/TBI_70_{timestamp}")
save_root.mkdir(parents=True, exist_ok=True)

# ── 1. CSV ───────────────────────────────────────────────────
df_save = pd.DataFrame(results_).drop(
    columns=[c for c in pd.DataFrame(results_).columns if c.startswith("_")],
    errors="ignore"
)
df_save.to_csv(save_root / "rotterdam_scores_.csv", index=False)
print(f"✅ rotterdam_scores_300.csv  ({len(df_save)} satır)")

# ── 2. Seçili slice'ları kopyala ─────────────────────────────
slices_root = save_root / "selected_slices"
slices_root.mkdir(exist_ok=True)

print(f"\n📂 70 hasta için seçili slice'lar kopyalanıyor...")

for i, rec in enumerate(results_):
    pid       = rec["pid"]
    pid_short = pid.split()[0]
    print(f"  [{i+1:2d}/70] {pid_short}", end=" ... ")

    try:
        patient_dir = find_patient_dir_by_pid(pid, CQ500_ROOT)
        if patient_dir is None:
            print("path yok"); continue

        # 5mm serisini bul
        all_dcm    = list(patient_dir.rglob("*.dcm"))
        series_map = {}
        for f in all_dcm:
            series_map.setdefault(f.parent, []).append(f)

        best_series = None
        for series_dir, files in series_map.items():
            try:
                ds = pydicom.dcmread(str(files[0]))
                if float(getattr(ds, "SliceThickness", 0)) >= 4.0:
                    best_series = (series_dir, sorted(files))
                    break
            except Exception:
                continue

        if best_series is None:
            print("5mm seri yok"); continue

        _, all_files = best_series

        # Pipeline'dan seçilen slice indekslerini al
        vol, px_sp = load_dicom_volume(patient_dir)
        sel_slices = ssa_.select_slices(vol)

        # Seçili slice'lara karşılık gelen DICOM dosyalarını kopyala
        out_dir = slices_root / pid_short
        out_dir.mkdir(exist_ok=True)

        copied = 0
        for idx in sel_slices:
            if idx < len(all_files):
                src = all_files[idx]
                dst = out_dir / f"slice_{idx:03d}_{src.name}"
                shutil.copy2(src, dst)
                copied += 1

        print(f"{copied} slice ✅  (indeksler: {sel_slices})")

    except Exception as e:
        print(f"❌ {e}")

# ── 3. Zip ───────────────────────────────────────────────────
print("\n📦 Zip'leniyor...")
shutil.make_archive(str(save_root), "zip", save_root)

print(f"\n{'='*55}")
print(f"✅ Tamamlandı!")
print(f"   CSV    : {save_root}/rotterdam_scores_70.csv")
print(f"   Slice  : {save_root}/selected_slices/<pid>/slice_NNN_*.dcm")
print(f"   Zip    : /kaggle/working/TBI_70_{timestamp}.zip")
print(f"   → Kaggle Output sekmesinden indir")

✅ rotterdam_scores_300.csv  (300 satır)

📂 70 hasta için seçili slice'lar kopyalanıyor...
  [ 1/70] CQ500CT0 ... 4 slice ✅  (indeksler: [16, 18, 20, 22])
  [ 2/70] CQ500CT101 ... 6 slice ✅  (indeksler: [10, 11, 13, 15, 17, 21])
  [ 3/70] CQ500CT102 ... 6 slice ✅  (indeksler: [8, 10, 13, 14, 16, 18])
  [ 4/70] CQ500CT103 ... 6 slice ✅  (indeksler: [10, 11, 15, 17, 19, 21])
  [ 5/70] CQ500CT104 ... 1 slice ✅  (indeksler: [10])
  [ 6/70] CQ500CT105 ... 6 slice ✅  (indeksler: [8, 9, 11, 13, 14, 18])
  [ 7/70] CQ500CT106 ... 6 slice ✅  (indeksler: [13, 14, 15, 19, 23, 24])
  [ 8/70] CQ500CT107 ... 5 slice ✅  (indeksler: [11, 12, 13, 14, 15])
  [ 9/70] CQ500CT108 ... 6 slice ✅  (indeksler: [17, 18, 19, 20, 21, 26])
  [10/70] CQ500CT109 ... 6 slice ✅  (indeksler: [13, 14, 15, 16, 17, 23])
  [11/70] CQ500CT110 ... 6 slice ✅  (indeksler: [15, 17, 21, 23, 25, 26])
  [12/70] CQ500CT111 ... 5 slice ✅  (indeksler: [15, 17, 18, 19, 20])
  [13/70] CQ500CT113 ... 1 slice ✅  (indeksler: [12])
  [14/70]

## ✅ Pipeline Özeti

| Component | Method | Implementation |
|-----------|--------|----------------|
| Slice Selection | SSA (Qi 2013) | ✅ Rule-based geometric |
| Ideal Midline | Skull symmetry | ✅ Exhaustive search |
| Deformed Midline | Ventricle detection | ✅ With U-Net integration |
| MLS Calculation | Geometric distance | ✅ Mean aggregation |
| Hemorrhage Detection | U-Net ResNet34 | ✅ Deep learning |
| Basal Cistern | CR (Toledo 2021) | ✅ K-means clustering |
| Rotterdam Score | Maas 2005 | ✅ Complete 2-6 scale |

### Yenilikçi Özellikler:
1. **Hybrid Architecture**: Deep learning (U-Net) + rule-based algorithms
2. **U-Net Integration**: Hemorrhage masks exclude false CSF detections
3. **Automated CR**: No manual annotation required for basal cistern
4. **End-to-end**: Single pipeline for complete Rotterdam Score

### Kaynaklar:
- **MLS**: Qi et al. (2013) - Automatic Midline Shift Detection
- **Basal Cistern**: Toledo et al. (2021) - Compression Ratio
- **Rotterdam Score**: Maas et al. (2005) - Prognostic Value
- **Dataset**: CQ500 - Qure.ai Head CT Dataset